In [1]:
from langchain_groq import ChatGroq
from dotenv import load_dotenv
import os 

from youtube_transcript_api import YouTubeTranscriptApi, TranscriptsDisabled
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS
from langchain_core.prompts import PromptTemplate
from langchain_google_genai import GoogleGenerativeAIEmbeddings

C:\Users\Dell\AppData\Local\Temp\ipykernel_17876\2699688135.py:7: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.vectorstores import FAISS


In [9]:
load_dotenv()


True

Step 1 Indexing 

In [3]:
#Indexing 
#Step 1 getting the transcript of the Vdo !

try:
    video_id = 'JinpVA6p8Mo' # this is the id of yt vdo
    transcript_list = YouTubeTranscriptApi().fetch(video_id, languages=["en"])

    #these transcripts are in the list of dictioneries that's why in the next code we 
    # are joining those 

    transcript = " ".join(chunk.text for chunk in transcript_list)
    print(transcript)

except TranscriptsDisabled:
    print('No captions avalable for this video')

(snowballs thud) (expectant music plays) - We recently took a
family vacation to the snow and right away I noticed two things. Number one: making and throwing snowballs with your hands is really inefficient. (Sara McLachlan's "In the
Arms of the Angel" plays) - And number two: as an uncle, you're always getting
unfairly ganged up on. ("In the Arms of the Angel" continues) (snowball thuds) - So in an attempt to
address both of these issues I decided to create a
snowball machine gun. (snowballs thump) (laughs maniacally) For starters, I needed
to find a power source. (upbeat music plays) Next, I need to find an awesome brother to help with the planning and fabrication. And you may remember Brian from the video where he Bob Vila-ed these LED edge lit tables into existence. (apprehensive music plays) (music becomes more upbeat) (duct tape squeaks) - Okay, so here is our first test shot to see if this concept is even gonna work. So the concept seemed to work. But since I'm always outnumbere

In [ ]:
# Chunking (Text Splitter )

splitter = RecursiveCharacterTextSplitter(chunk_size = 1000,chunk_overlap = 200)
chunks = splitter.create_documents([transcript])

print(len(chunks))

4


In [ ]:
# Embedding and vectore storing using FIASS(by meta

embeddings = GoogleGenerativeAIEmbeddings(
    model="gemini-embedding-001"
)
vector_store = FAISS.from_documents(chunks,embeddings)
vector_store.index_to_docstore_id

# this is the code to see how the chunks embedded id looks in the vector data base
# and it's id(done using gpt model's)

for vector_id, doc_id in vector_store.index_to_docstore_id.items():
    print(f"Vector ID: {vector_id} | Document ID: {doc_id}")

# Indexing part is completed 

Vector ID: 0 | Document ID: 1db07767-06ea-48e1-880d-75b14ade688a
Vector ID: 1 | Document ID: 71aa6fcc-8f61-4c63-b215-2bded5b37550
Vector ID: 2 | Document ID: 605a8321-17ab-44e2-99de-1307352ceb0e
Vector ID: 3 | Document ID: 187e9148-482d-4e18-a450-0deded3a8408


Step 2 Retrieval

In [ ]:
retriever = vector_store.as_retriever(search_type = "similarity", search_kwargs ={"k":4}) #k means number of documents  and similarty is thye pf search 

Step 3 Augmentation

In [10]:
llm = ChatGroq(model="openai/gpt-oss-20b",temperature= 0.2)

In [19]:
prompt = PromptTemplate(
    template= """
    You are a helpful assistent.
    Answer ONLY from the provided transcript context.
    if the context is insufficient, just say you don't know.

    {context}
    question : {question}
    """,
    input_variables={'context','question'}
)

In [29]:
question = 'Are we making a iceball in this vdo if not then what are we making tell me in 200 words'
retrieved_docs = retriever.invoke(question)

In [30]:
retrieved_docs

[Document(id='1db07767-06ea-48e1-880d-75b14ade688a', metadata={}, page_content='(snowballs thud) (expectant music plays) - We recently took a\nfamily vacation to the snow and right away I noticed two things. Number one: making and throwing snowballs with your hands is really inefficient. (Sara McLachlan\'s "In the\nArms of the Angel" plays) - And number two: as an uncle, you\'re always getting\nunfairly ganged up on. ("In the Arms of the Angel" continues) (snowball thuds) - So in an attempt to\naddress both of these issues I decided to create a\nsnowball machine gun. (snowballs thump) (laughs maniacally) For starters, I needed'),
 Document(id='605a8321-17ab-44e2-99de-1307352ceb0e', metadata={}, page_content="concept's pretty simple. You just load up to 15 snowballs in this sort of magazine\nand then clip it on. And then use the trigger\nto control the airflow which determines if you want\nto shoot them hard or soft. And then this plunger\ncontrols the rate of fire. So you can do a sing

In [31]:
context_text = "\n \n".join(doc.page_content for doc in retrieved_docs)

In [32]:
context_text

'(snowballs thud) (expectant music plays) - We recently took a\nfamily vacation to the snow and right away I noticed two things. Number one: making and throwing snowballs with your hands is really inefficient. (Sara McLachlan\'s "In the\nArms of the Angel" plays) - And number two: as an uncle, you\'re always getting\nunfairly ganged up on. ("In the Arms of the Angel" continues) (snowball thuds) - So in an attempt to\naddress both of these issues I decided to create a\nsnowball machine gun. (snowballs thump) (laughs maniacally) For starters, I needed\n \nconcept\'s pretty simple. You just load up to 15 snowballs in this sort of magazine\nand then clip it on. And then use the trigger\nto control the airflow which determines if you want\nto shoot them hard or soft. And then this plunger\ncontrols the rate of fire. So you can do a single shot or semi-automatic or just unload them all. And when you\'re out of ammo you can just reload a new magazine in less than three seconds. (music continu

In [33]:
final_promt = prompt.invoke({"context" : context_text, "question":question})

In [34]:
final_promt

StringPromptValue(text='\n    You are a helpful assistent.\n    Answer ONLY from the provided transcript context.\n    if the context is insufficient, just say you don\'t know.\n\n    (snowballs thud) (expectant music plays) - We recently took a\nfamily vacation to the snow and right away I noticed two things. Number one: making and throwing snowballs with your hands is really inefficient. (Sara McLachlan\'s "In the\nArms of the Angel" plays) - And number two: as an uncle, you\'re always getting\nunfairly ganged up on. ("In the Arms of the Angel" continues) (snowball thuds) - So in an attempt to\naddress both of these issues I decided to create a\nsnowball machine gun. (snowballs thump) (laughs maniacally) For starters, I needed\n \nconcept\'s pretty simple. You just load up to 15 snowballs in this sort of magazine\nand then clip it on. And then use the trigger\nto control the airflow which determines if you want\nto shoot them hard or soft. And then this plunger\ncontrols the rate of 

Step 4 Generation 

In [37]:
answer = llm.invoke(final_promt)
print(answer.content)

No, the video does not describe making an iceball. Instead, it focuses on a snowball machine gun. The narrator explains that traditional hand‑thrown snowballs are inefficient and that, as an uncle, he often gets ganged up on. To solve both problems, he builds a device that can load up to fifteen snowballs into a magazine, clip it onto a pack, and fire them using a trigger that controls airflow for soft or hard shots. A plunger sets the rate of fire, allowing single shots, semi‑automatic bursts, or a full unload. The device is portable, reloadable in under three seconds, and even looks like a “Ghostbuster” gadget. The video includes test shots, humor, and a mention of a PDF with detailed instructions. No mention of iceballs or melting snow is made; the focus remains on snowballs and the machine gun’s mechanics. Thus, the project is a snowball launcher, not an iceball maker. The machine is portrayed as a playful invention for family fun, with the narrator joking about looking like a Ghos

RAG process is completed now we are building chains 

In [38]:
from langchain_core.runnables import RunnableParallel,RunnablePassthrough,RunnableLambda
from langchain_core.output_parsers import StrOutputParser

In [39]:
def format_docs(retrieved_docs):
    context_text = "\n\n".join(doc.page_content for doc in retrieved_docs)
    return context_text

In [47]:
parallel_chain = RunnableParallel({
    'context': retriever | RunnableLambda(format_docs),
    'question': RunnablePassthrough()
})

In [ ]:
parallel_chain.invoke("who is the main target ? ")

{'context': 'concept\'s pretty simple. You just load up to 15 snowballs in this sort of magazine\nand then clip it on. And then use the trigger\nto control the airflow which determines if you want\nto shoot them hard or soft. And then this plunger\ncontrols the rate of fire. So you can do a single shot or semi-automatic or just unload them all. And when you\'re out of ammo you can just reload a new magazine in less than three seconds. (music continues) (snowballs crash and smack) - It works really well, and with the pack on\nyour back it\'s portable so you can use it wherever. Plus, it makes you look like\na freaking Ghost Buster. (Ghost Buster theme music plays) - Yeah! (snowball machine whirring) (laughs) - Yeah! (Ghost Buster music resumes) (snowball machine shoots) (snowballs thud) (snowballs smacking rapidly) - Wait, wait, wait, wait wait. Why would you wanna shoot me? - Because.\n\n- Just think about it. - It\'s payback, my friend. - But I\'m such a nice uncle. I don\'t deserve i

In [50]:
parser = StrOutputParser()

In [ ]:
main_chain = parallel_chain | prompt | llm | parser

In [59]:
main_chain.invoke("summerize this whole vdo ?  ")

'The video is a DIY tutorial in which the creator explains how they built a “snowball machine gun.” They start by noting two problems: throwing snowballs by hand is inefficient, and as an uncle they’re often ganged‑up on. To solve this, they design a device that can hold up to 15 snowballs in a magazine, clip it on, and fire them using a trigger that controls airflow (for hard or soft shots). A plunger sets the rate of fire, allowing single, semi‑automatic, or full‑auto modes, and a new magazine can be loaded in under three seconds. The creator mentions a PDF with detailed instructions available for free in the video description, and credits a brother (who previously helped with LED tables) for help. The video includes playful sound effects, upbeat music, and references to Ghostbusters, ending with a comedic exchange about why someone would want to shoot them.'